# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the **Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya** using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema and can be accessed via the following URL.

In [ ]:
# Install the mlcroissant library if not already installed
!pip install --quiet mlcroissant

## 1. Data Loading
Let's load the Croissant schema and dataset metadata using `mlcroissant`. The dataset metadata exposes rich semantic information, and the `@id` fields are the recommended way to reference all entities.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to the Croissant JSON-LD schema
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name)
print(metadata.description)

## 2. Data Overview
List available record sets, their `@id`s, fields, and details using the Croissant metadata. All references to entities (record sets, fields, columns) are shown by their `@id`.

> **Note:** If a dataset's Croissant schema is well-formed, `.record_sets` and `.fields` will enumerate the contents. For educational purposes, we programmatically enumerate the accessible record sets and fields with their `@id`s.

In [ ]:
# Display all available record sets by @id
print("Available Record Sets:")
record_sets = [rs for rs in getattr(metadata, "record_sets", [])]
for rs in record_sets:
    print(f"- Record Set: {rs['@id']} | Name: {rs.get('name', '')}")

if not record_sets:
    print("No record sets found directly via metadata. Loading them via dataset...")

# Fallback: list all possible record sets using dataset's internal catalog
record_set_ids = dataset.record_set_ids
for rs_id in record_set_ids:
    info = dataset.get_record_set(rs_id)
    name = info.get('name', '')
    print(f"- Record Set @id: {rs_id} | Name: {name}")
    fields = info.get('fields', [])
    for field in fields:
        print(f"    - Field @id: {field['@id']} | Name: {field.get('name', '')} | Data type: {field.get('data_type', '')}")

## 3. Data Extraction
We'll now extract the tabular data from the dataset by iterating through all record set `@id`s (as listed above) and loading each into a pandas DataFrame.

> The data is referenced by **record set @id** and **field @id** for maximum reproducibility and schema-alignment.

In [ ]:
dataframes = {}

# Use discovered record_set_ids above
if not record_set_ids:
    print("No Record Sets found in dataset; cannot extract data.")
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                dataframes[record_set_id] = pd.DataFrame(records)
                print(f"Loaded DataFrame for record set @id: {record_set_id} | Shape: {dataframes[record_set_id].shape}")
            else:
                print(f"No records found for record set @id: {record_set_id}")
        except Exception as e:
            print(f"Error loading records for {record_set_id}: {e}")

    if dataframes:
        # Show an example: pick the first available DataFrame
        demo_rsid = next(iter(dataframes.keys()))
        print(f"\nColumns in record set {demo_rsid}:\n", dataframes[demo_rsid].columns.tolist())
        display(dataframes[demo_rsid].head())

## 4. Exploratory Data Analysis (EDA)
Apply data filtering, normalization, grouping, and summary operations, referencing all fields by their `@id` as per the Croissant specification.

> Select a numeric field of interest and another groupable field from the overview above. Adjust threshold and any field references as needed for your own analysis.

In [ ]:
# Example: Find a numeric field and a group field for demo EDA.
import numpy as np
if dataframes:
    # Use DataFrame loaded above
    rsid = demo_rsid
    df = dataframes[rsid]

    # Find a numeric column (field @id)
    numeric_fields = df.select_dtypes(include=["number"]).columns.tolist()
    group_fields = df.select_dtypes(include=["object", "category"]).columns.tolist()

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field: {numeric_field_id}")
        threshold = np.nanpercentile(df[numeric_field_id].dropna(), 50)  # Median as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (median): {filtered_df.shape[0]} rows")

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Sample normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if groupable, show group-wise summaries.

In [ ]:
# Visualization using matplotlib or seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping is possible, show barplot
    if group_fields:
        plt.figure(figsize=(10,4))
        # Use top 10 groups for visibility
        order = grouped_df.sort_values(f"mean_{numeric_field_id}", ascending=False)[group_field_id][:10]
        sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}", order=order)
        plt.title(f"Group-wise Mean of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- This notebook has illustrated how to use the Croissant schema and `mlcroissant` library to load, process, and analyze a dataset using only `@id` references for full reproducibility.
- You can now continue with deeper statistical or machine learning analyses relevant to your research context.